# 03 · The "paper way" vs the honest way

**Goal:** many published studies cut each recording into short windows (here 4 s), shuffle *all* windows together, and then split them into training and test sets. Windows from the same child then end up on both sides. Here we copy that approach and compare it with an **honest split by child**. Both use the same features and the same model.

**How it fits the project:** this is the core question. Do the high published accuracies survive a child-level split? Any gap between the two scores is **leakage**: the model has partly seen the test children during training.

Analogy: an exam whose questions are slightly reworded homework questions. A high score shows memory of the homework, not understanding.

**Question:** How many 4-second windows do we get, and how many per group?

**How:**
1. For each child, `window_features` (in `src/features.py`):
   - band-pass filters the whole recording at 1–40 Hz,
   - cuts it into non-overlapping 4-second pieces (512 samples) and drops the leftover at the end,
   - computes the same 76 features as notebook 02 for each piece. With `n_fft=512` each piece is one Welch segment, so each spectrum is noisier than the per-child one.
2. Add the child's `child_id` and `label` to every window row. `pd.concat` stacks all children into one big table.
3. `X` = features, `y` = labels, `g` = **groups** (child IDs). Groups tell scikit-learn which windows belong to the same child.

**What we found:** 4173 windows × 78 columns (76 features + `child_id` + `label`). **Control 1843, ADHD 2330.** ADHD has more windows because its recordings are longer (notebook 01). Always guessing "ADHD" would already score 2330 ÷ 4173 = **0.56 accuracy**, the baseline for accuracy here.

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.load import list_children, load_child
from src.features import window_features

parts = []
for _, k in pd.DataFrame(list_children()).iterrows():
    f = window_features(load_child(k.path))
    f["child_id"], f["label"] = k.child_id, k.label
    parts.append(f)
win = pd.concat(parts, ignore_index=True)

X, y, g = win.drop(columns=["child_id", "label"]), win.label, win.child_id
print(win.shape)
print(win.groupby("label").size())

(4173, 78)
label
0    1843
1    2330
dtype: int64


**Question:** Does the way we split the data change the score?

**How:**
1. **Random forest**: 300 decision trees. Each tree is trained on a random sample of rows and features, and they vote. `n_jobs=-1` uses all CPU cores.
2. Two ways to split into 5 folds:
   - `StratifiedKFold(shuffle=True)`, the **paper way**: windows are shuffled at random, keeping the ADHD/Control mix. It ignores which child a window came from, so each child's windows land in both training and test data. The warning *"The groups parameter is ignored by StratifiedKFold"* confirms that it does not look at children.
   - `StratifiedGroupKFold(shuffle=True)`, the **honest way**: all windows of a child stay together in one fold, while it tries to keep the class mix balanced. A child is either fully in training or fully in test.
3. `cross_validate` returns accuracy and AUC for each fold. We print the means.

**What we found:** | split | accuracy | AUC |
|---|---|---|
| paper way (windows shuffled) | **0.79** | **0.88** |
| honest (split by child) | **0.63** | **0.64** |

Same data, same features, same model; only the split differs. The drop (0.16 accuracy, 0.24 AUC) is **leakage**. In the paper way, the model trains on other windows of the very same child and can recognise them (see the last cell). Honest accuracy (0.63) is only a little above the 0.56 always-ADHD baseline. Honest AUC (0.64) is about the same as recording length alone (0.65, notebook 01).

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
splits = {
    "PAPER WAY (pieces shuffled)": StratifiedKFold(5, shuffle=True, random_state=42),
    "HONEST (split by child)":     StratifiedGroupKFold(5, shuffle=True, random_state=42),
}
for name, cv in splits.items():
    r = cross_validate(model, X, y, groups=g, cv=cv, scoring=["accuracy", "roc_auc"])
    print(f"{name:28s} accuracy {r['test_accuracy'].mean():.2f}   AUC {r['test_roc_auc'].mean():.2f}")

c:\Users\Rajesh Sampathkumar\miniconda3\envs\medimg\Lib\site-packages\sklearn\model_selection\_split.py:884: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


PAPER WAY (pieces shuffled)  accuracy 0.79   AUC 0.88
HONEST (split by child)      accuracy 0.63   AUC 0.64


**Question:** If we combine all windows of a child into one score per child, how good is the honest model?

**How:**
1. `cross_val_predict` gives every window a prediction from the fold in which its child was in the **test** set, so it stays honest.
2. `method="predict_proba"` returns probabilities. `[:, 1]` is the probability of ADHD.
3. Group the windows by child and **average** their probabilities, giving one score per child. This is like averaging several measurements from the same patient.
4. `roc_auc_score` computes AUC over the 121 children.

**What we found:** **Per-child AUC 0.70.** This is higher than the window-level 0.64, because averaging smooths out noisy single windows. It is still honest. However, it comes from **one** 5-fold split (one `random_state`) of only 121 children, and a different split could move it by several points. So we cannot yet say it really beats recording length alone (0.65). Repeating it with several random splits would tell.

In [3]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

honest = StratifiedGroupKFold(5, shuffle=True, random_state=42)
p = cross_val_predict(model, X, y, groups=g, cv=honest, method="predict_proba")[:, 1]
per_child = pd.DataFrame({"child_id": g, "label": y, "p": p}).groupby("child_id").mean()
print("Honest, per-child AUC:", round(roc_auc_score(per_child.label, per_child.p), 2))

Honest, per-child AUC: 0.7


**Question:** Why does the paper way score so high? Can the model tell *which child* a window came from?

**How:**
1. Same random forest, but the target is now `g` (the child ID) instead of ADHD/Control, so there are 121 possible answers.
2. Use the paper-style shuffled split, so every child has windows in both training and test data.
3. Accuracy = fraction of test windows assigned to the correct child. Random guessing would give 1 ÷ 121 ≈ 0.008.

**What we found:** **0.70**, about 85 times better than guessing. Each child's EEG carries a strong personal **"fingerprint"**, like a voice you recognise on the phone. Every child has one fixed label, so recognising the child is enough to know the label. This is how leakage inflates the paper-way score: the model can recognise *who* it is rather than learn what ADHD looks like.

In [4]:
from sklearn.model_selection import cross_val_score
who = cross_val_score(model, X, g, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy")
print(f"Who-is-this accuracy: {who.mean():.2f}  (guessing = 0.008)")

Who-is-this accuracy: 0.70  (guessing = 0.008)


## Key takeaways
- Same data, features and model: paper way **0.79 accuracy / 0.88 AUC**; honest child-level split **0.63 / 0.64**.
- The gap is leakage. With shuffled windows, the same child appears in both training and test data.
- A model can identify the child from a single 4-second window 70% of the time (chance 0.8%). This strong fingerprint explains the leakage.
- The honest window-level AUC (0.64) is close to recording length alone (0.65). Averaging per child gives 0.70, but from a single split.
- **Answer so far:** on this dataset, the high accuracies from shuffled windows do **not** survive a child-level split.

## Open questions
- Does the per-child AUC of 0.70 hold over repeated random splits, and does it reliably beat length (0.65)?
- How much do recording length and window count (more windows for ADHD) affect the honest result? Try equal length or an equal number of windows per child.
- Is the child "fingerprint" driven by artifacts (T8 muscle, eye movements) or by genuine brain rhythms?
- Would a deep-learning model on raw windows, as many papers use, show an even bigger gap?